In [95]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn import metrics
from sklearn.neighbors import KNeighborsRegressor
from sklearn import tree
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error



In [96]:
df = pd.read_csv('cleaneddata.csv')

In [97]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1274 entries, 0 to 1273
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Company      1274 non-null   object 
 1   TypeName     1274 non-null   object 
 2   Ram          1274 non-null   int64  
 3   OpSys        1274 non-null   object 
 4   Weight       1274 non-null   float64
 5   Price        1274 non-null   float64
 6   CPU          1274 non-null   object 
 7   HDD          559 non-null    float64
 8   SSD          837 non-null    float64
 9   GPU          1274 non-null   object 
 10  TouchScreen  1274 non-null   int64  
 11  IPS          1274 non-null   int64  
 12  PPI          1274 non-null   float64
dtypes: float64(5), int64(3), object(5)
memory usage: 129.5+ KB


In [98]:
df.head()

,Company,TypeName,Ram,OpSys,Weight,Price,CPU,HDD,SSD,GPU,TouchScreen,IPS,PPI
0,Apple,Ultrabook,8,Mac,1.37,71378.6832,Intel Core i5,NaN,128.0,Intel,0,1,226.983005
1,Apple,Ultrabook,8,Mac,1.34,47895.5232,Intel Core i5,NaN,NaN,Intel,0,0,127.677940
2,HP,Notebook,8,Other,1.86,30636.0000,Intel Core i5,NaN,256.0,Intel,0,0,141.211998
3,Apple,Ultrabook,16,Mac,1.83,135195.3360,Intel Core i7,NaN,512.0,AMD,0,1,220.534624
4,Apple,Ultrabook,8,Mac,1.37,96095.8080,Intel Core i5,NaN,256.0,Intel,0,1,226.983005


In [99]:
y = np.log(df['Price'])
X = df.drop(['Price'],axis = 1)

In [100]:

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.15,
                                                    random_state=2)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(1082, 12) (192, 12) (1082,) (192,)


In [101]:
mapper = {i:value for i,value in enumerate(X_train.columns)}
mapper

{0: 'Company',
 1: 'TypeName',
 2: 'Ram',
 3: 'OpSys',
 4: 'Weight',
 5: 'CPU',
 6: 'HDD',
 7: 'SSD',
 8: 'GPU',
 9: 'TouchScreen',
 10: 'IPS',
 11: 'PPI'}

In [102]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1274 entries, 0 to 1273
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Company      1274 non-null   object 
 1   TypeName     1274 non-null   object 
 2   Ram          1274 non-null   int64  
 3   OpSys        1274 non-null   object 
 4   Weight       1274 non-null   float64
 5   Price        1274 non-null   float64
 6   CPU          1274 non-null   object 
 7   HDD          559 non-null    float64
 8   SSD          837 non-null    float64
 9   GPU          1274 non-null   object 
 10  TouchScreen  1274 non-null   int64  
 11  IPS          1274 non-null   int64  
 12  PPI          1274 non-null   float64
dtypes: float64(5), int64(3), object(5)
memory usage: 129.5+ KB


In [103]:
df['HDD'] = pd.to_numeric(df['HDD'], errors='coerce').fillna(0).astype(float)
df= df.drop('HDD', axis=1)
df['SSD'] = pd.to_numeric(df['SSD'], errors='coerce').fillna(0).astype(float)


categorical_features_for_encoding = [
    'Company',
    'TypeName',
    'OpSys',
    'CPU',
    'GPU', 
]

numerical_features_for_scaling = [
    'Ram', 
    'Weight',
    'TouchScreen',
    'IPS',
    'PPI',
    'SSD'
    

]

In [104]:
from sklearn import metrics

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features_for_encoding),

        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numerical_features_for_scaling)
    ]
)

In [105]:
step2_knn = KNeighborsRegressor(n_neighbors=5)

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', step2_knn)
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)

print('R2 score:', metrics.r2_score(y_test, y_pred))
print('MAE:', metrics.mean_absolute_error(y_test, y_pred))

R2 score: 0.8270912039661974
MAE: 0.18676294792222067


In [106]:
import pickle

pickle.dump(df,open('df.pkl','wb'))

In [107]:
import joblib
joblib.dump(pipe, 'lapdata.pkl')

['lapdata.pkl']

In [108]:
X_train.head()

,Company,TypeName,Ram,OpSys,Weight,CPU,HDD,SSD,GPU,TouchScreen,IPS,PPI
21,Lenovo,Gaming,8,Windows,2.50,Intel Core i5,1000.0,128.0,Nvidia,0,1,141.211998
790,Asus,Gaming,8,Windows,2.24,Intel Core i7,1000.0,128.0,Nvidia,0,0,141.211998
273,Lenovo,2 in 1 Convertible,16,Windows,1.36,Intel Core i7,NaN,512.0,Intel,1,0,209.800683
397,Lenovo,Notebook,8,Windows,1.90,Intel Core i5,NaN,256.0,Intel,0,1,157.350512
921,HP,Ultrabook,8,Windows,1.84,Intel Core i7,NaN,256.0,AMD,0,0,141.211998


In [109]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# Save the test set to files
X_train.to_csv("splits/X_train.csv", index=False)
X_test.to_csv("splits/X_test.csv", index=False)
y_train.to_csv("splits/y_train.csv", index=False)
y_test.to_csv("splits/y_test.csv", index=False)

In [110]:
y_pred = pipe.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

RMSE: 0.2064
R² Score: 0.8883


In [111]:
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100  # In %

# Adjusted R²
n = X_test.shape[0]
p = X_test.shape[1]
adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))

print(f"Adjusted R² Score: {adjusted_r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE: {mape:.2f}%")

Adjusted R² Score: 0.8828
MAE: 0.1598
MAPE: 1.46%


In [112]:
metrics = {
    "Metric": ["RMSE", "R²", "Adjusted R²", "MAE", "MAPE"],
    "Value": [rmse, r2, adjusted_r2, mae, mape]
}
results_df = pd.DataFrame(metrics)
print(results_df)

        Metric     Value
0         RMSE  0.206368
1           R²  0.888324
2  Adjusted R²  0.882787
3          MAE  0.159754
4         MAPE  1.461827


In [113]:
results_df.to_csv("summaryOFresults.csv", index=False)